# DS 301 Final Project: Credit Card Default Prediction
**Team Members:** Chung Vong (Simon), Maria, Eduardo, Gabriel  
## Reproduction of the Selected Article and Logistic Regression Contribution

**Scientific basis:** *Predicting Default of Credit Card Clients Using Three Supervised Machine Learning Algorithms* — Thomson Ly, Rene Schuller, Katarina Simanic, and Sukhpreet Singh.

**Dataset:** Default of Credit Card Clients from the UCI Machine Learning Repository (30,000 clients from Taiwan).

**Models reproduced from the article:** Decision Tree, K-Nearest Neighbors (KNN), and Support Vector Machine (SVM).

**Group contribution:** Logistic Regression, tuned and evaluated with the same cleaned dataset.

The target is binary:

- **0:** the client does not default next month.
- **1:** the client defaults next month.

For this project, precision, recall, and F1-score refer to the **default class (1)**.

## 1. Article Methodology

The selected article follows these main steps:

1. Remove the ID column.
2. Remove undocumented values from EDUCATION and MARRIAGE.
3. Split the data into 80% training and 20% testing.
4. Standardize the features using the training data.
5. Apply random oversampling only to the training data.
6. Train Decision Tree, KNN, and SVM models.
7. Compare Accuracy, Precision, Recall, and F1-score.

| Article model | Parameters reported as best |
|---|---|
| Decision Tree | criterion=entropy, max_depth=None, max_features=9, splitter=best |
| KNN | n_neighbors=1, metric=euclidean, weights=uniform in the final experiment |
| SVM | kernel=rbf, C=10 |

The article considers F1-score important because the dataset is imbalanced and the default class is the minority class.

## 2. Import the Libraries

The Logistic Regression import is marked clearly because it is the new model added by the group.

In [1]:
# Import the libraries used in the project
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression  # GROUP CONTRIBUTION
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 3. Load and Understand the Dataset

The local dataset is used first so the notebook is reproducible without an internet connection. If the local file is not available, the code tries to download UCI dataset ID 350.

In [4]:
# Install the ucimlrepo library if it's not already installed
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    !pip install ucimlrepo
    from ucimlrepo import fetch_ucirepo # Try importing again after installation

# Load the local dataset first and use UCI as a fallback
possible_paths = [
    Path("data/default_of_credit_card_clients.xls"),
    Path("../data/default_of_credit_card_clients.xls")
]
local_file = next((path for path in possible_paths if path.exists()), None)

if local_file is not None:
    raw_data = pd.read_excel(local_file, header=1)
    X = raw_data.drop(columns=["ID", "default payment next month"]).copy()
    y = raw_data["default payment next month"].astype(int).copy()
else:
    credit_data = fetch_ucirepo(id=350)
    X = credit_data.data.features.copy()
    y = credit_data.data.targets.squeeze().astype(int).copy()

# Use the variable names presented in the selected article
X.columns = [
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]
y.name = "DEFAULT"

print(f"Rows: {len(X):,}")
print(f"Features: {X.shape[1]}")
print(f"Missing values: {X.isna().sum().sum()}")
print(f"Default rate: {y.mean():.2%}")
print(X.head().to_string(index=False))

Rows: 30,000
Features: 23
Missing values: 0
Default rate: 22.12%
 LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6
     20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0
    120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000
     90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000
     50000    2          2         1   37      0      0      0      0      0      0      46990 

In [5]:
# Remove undocumented categories exactly as described in the article
valid_rows = (
    X["EDUCATION"].isin([1, 2, 3, 4])
    & X["MARRIAGE"].isin([1, 2, 3])
)

removed_rows = int((~valid_rows).sum())
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

class_distribution = pd.DataFrame({
    "Class": ["No default (0)", "Default (1)"],
    "Clients": [(y == 0).sum(), (y == 1).sum()],
    "Percentage": [(y == 0).mean(), (y == 1).mean()]
})

print(f"Removed undocumented rows: {removed_rows}")
print(f"Remaining rows: {len(X):,}")
print(class_distribution.round(3).to_string(index=False))

Removed undocumented rows: 399
Remaining rows: 29,601
         Class  Clients  Percentage
No default (0)    22996       0.777
   Default (1)     6605       0.223


### Why oversampling is necessary

Only about 22% of the clients are in the default class. A model could obtain a reasonable accuracy by predicting mostly non-default cases. Random oversampling gives the minority class more representation during training.

In [6]:
# Split, standardize, and oversample the training data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# Fit the scaler only on the training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create a balanced training set using random oversampling
training_data = pd.DataFrame(X_train_scaled)
training_data["DEFAULT"] = y_train.to_numpy()

majority_class = training_data[training_data["DEFAULT"] == 0]
minority_class = training_data[training_data["DEFAULT"] == 1]
minority_oversampled = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=RANDOM_STATE
)

balanced_training = pd.concat([majority_class, minority_oversampled])
balanced_training = balanced_training.sample(frac=1, random_state=RANDOM_STATE)

y_train_balanced = balanced_training.pop("DEFAULT").astype(int).to_numpy()
X_train_balanced = balanced_training.to_numpy()

print(f"Training rows before oversampling: {len(y_train):,}")
print(f"Training rows after oversampling:  {len(y_train_balanced):,}")
print(pd.Series(y_train_balanced).value_counts().sort_index().to_string())

Training rows before oversampling: 23,680
Training rows after oversampling:  36,792
0    18396
1    18396


## 4. Results Published in the Selected Article

The following values are copied from the experimental-results tables in the selected article. Precision, recall, and F1-score refer to class 1, which represents default.

| Model | Accuracy | Precision (Default) | Recall (Default) | F1 (Default) |
|---|---:|---:|---:|---:|
| Decision Tree | 0.740 | 0.420 | 0.410 | 0.410 |
| KNN | 0.730 | 0.400 | 0.380 | 0.390 |
| SVM | **0.760** | **0.480** | **0.580** | **0.520** |

SVM is the strongest model reported in the article, but its default-class F1-score is still only 0.52.

In [7]:
# Store the published results for a direct comparison later
published_results = pd.DataFrame({
    "Model": ["Decision Tree", "KNN", "SVM"],
    "Accuracy": [0.74, 0.73, 0.76],
    "Precision": [0.42, 0.40, 0.48],
    "Recall": [0.41, 0.38, 0.58],
    "F1": [0.41, 0.39, 0.52]
})

print(published_results.round(3).to_string(index=False))

        Model  Accuracy  Precision  Recall   F1
Decision Tree      0.74       0.42    0.41 0.41
          KNN      0.73       0.40    0.38 0.39
          SVM      0.76       0.48    0.58 0.52


## 5. Reproduction of the Three Article Models

The code below uses the best parameters reported by the article. A fixed random seed and stratified split are included because the article does not provide its original random split seed. Small result differences are therefore expected.

In [8]:
# Calculate metrics for the default class
def evaluate_default_model(model_name, model, X_fit, y_fit, X_evaluate, y_evaluate):
    start_time = time.perf_counter()
    model.fit(X_fit, y_fit)
    predictions = model.predict(X_evaluate)
    elapsed_time = time.perf_counter() - start_time

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_evaluate, predictions),
        "Precision": precision_score(y_evaluate, predictions, zero_division=0),
        "Recall": recall_score(y_evaluate, predictions, zero_division=0),
        "F1": f1_score(y_evaluate, predictions, zero_division=0),
        "Seconds": elapsed_time,
        "Predictions": predictions
    }

In [9]:
# Reproduce the three models tested in the selected article
article_models = {
    "Decision Tree": DecisionTreeClassifier(
        criterion="entropy",
        max_depth=None,
        max_features=9,
        splitter="best",
        random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=1,
        weights="uniform",
        metric="euclidean"
    ),
    "SVM": SVC(
        C=10,
        kernel="rbf",
        cache_size=2000,
        random_state=RANDOM_STATE
    )
}

reproduction_details = []
for model_name, model in article_models.items():
    result = evaluate_default_model(
        model_name,
        model,
        X_train_balanced,
        y_train_balanced,
        X_test_scaled,
        y_test
    )
    reproduction_details.append(result)

reproduction_results = pd.DataFrame([
    {key: value for key, value in result.items() if key != "Predictions"}
    for result in reproduction_details
])

print(reproduction_results.round(3).to_string(index=False))

        Model  Accuracy  Precision  Recall    F1  Seconds
Decision Tree     0.735      0.404   0.399 0.401    1.020
          KNN     0.723      0.379   0.377 0.378    4.425
          SVM     0.757      0.464   0.566 0.510  139.161


### Reproduction check

| Model | Article F1 | Reproduced F1 | Absolute difference |
|---|---:|---:|---:|
| Decision Tree | 0.410 | 0.401 | 0.009 |
| KNN | 0.390 | 0.378 | 0.012 |
| SVM | 0.520 | 0.510 | 0.010 |

The reproduced results are close to the published results. This supports that the main preprocessing and modeling steps were reproduced correctly.

# 6. GROUP CONTRIBUTION — LOGISTIC REGRESSION

## This is the model added by our group

The selected article experimentally compares Decision Tree, KNN, and SVM. Our group adds **Logistic Regression** as a fourth classification model, as requested by the DS 301 project contribution requirements.

Why Logistic Regression?

- It is a clear baseline for binary classification.
- It is easier to explain than SVM.
- It can estimate the probability of default.
- It lets us test whether a simpler linear model can identify more default cases.

The next cell explicitly creates, tunes, trains, and evaluates Logistic Regression. It is intentionally separate from the loop used for the article models.

In [10]:
# ================================================================
# GROUP CONTRIBUTION: LOGISTIC REGRESSION
# ================================================================

# Create the Logistic Regression model added by our group
logistic_regression = LogisticRegression(
    max_iter=3000,
    solver="liblinear",
    random_state=RANDOM_STATE
)

# Test different regularization values for Logistic Regression
logistic_parameter_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

# Use stratified cross-validation and optimize the default-class F1-score
logistic_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

logistic_search = GridSearchCV(
    estimator=logistic_regression,
    param_grid=logistic_parameter_grid,
    scoring="f1",
    cv=logistic_cv,
    n_jobs=-1,
    refit=True
)

# Train Logistic Regression
logistic_start = time.perf_counter()
logistic_search.fit(X_train_balanced, y_train_balanced)

# Make predictions with Logistic Regression
logistic_predictions = logistic_search.predict(X_test_scaled)
logistic_seconds = time.perf_counter() - logistic_start

# Calculate the Logistic Regression results
logistic_result = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, logistic_predictions),
    "Precision": precision_score(y_test, logistic_predictions, zero_division=0),
    "Recall": recall_score(y_test, logistic_predictions, zero_division=0),
    "F1": f1_score(y_test, logistic_predictions, zero_division=0),
    "Seconds": logistic_seconds,
    "Best C": logistic_search.best_params_["C"]
}])

print("Best Logistic Regression parameters:", logistic_search.best_params_)
print(logistic_result.round(3).to_string(index=False))

Best Logistic Regression parameters: {'C': 0.01}
              Model  Accuracy  Precision  Recall    F1  Seconds  Best C
Logistic Regression     0.696       0.39   0.645 0.486    6.397    0.01


### Logistic Regression result

| Group contribution | Accuracy | Precision (Default) | Recall (Default) | F1 (Default) | Best C |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.696 | 0.390 | **0.645** | 0.486 | 0.01 |

The Logistic Regression model finds a larger percentage of the clients who actually default. Its recall is 0.645, but this comes with more false positive predictions and lower precision.

In [11]:
# Show the Logistic Regression confusion matrix as a clear table
logistic_matrix = confusion_matrix(y_test, logistic_predictions)
logistic_confusion_table = pd.DataFrame(
    logistic_matrix,
    index=["Actual: No default", "Actual: Default"],
    columns=["Predicted: No default", "Predicted: Default"]
)

print(logistic_confusion_table.to_string())

                    Predicted: No default  Predicted: Default
Actual: No default                   3267                1333
Actual: Default                       469                 852


## 7. Final Comparative Table

This table separates the published article results, our reproduction, and the Logistic Regression contribution.

In [12]:
# Build one final table with all results
published_table = published_results.copy()
published_table.insert(0, "Role", "Published in article")

reproduced_table = reproduction_results[
    ["Model", "Accuracy", "Precision", "Recall", "F1"]
].copy()
reproduced_table.insert(0, "Role", "Our reproduction")

logistic_table = logistic_result[
    ["Model", "Accuracy", "Precision", "Recall", "F1"]
].copy()
logistic_table.insert(0, "Role", "GROUP CONTRIBUTION")

final_comparison = pd.concat(
    [published_table, reproduced_table, logistic_table],
    ignore_index=True
)

print(final_comparison.round(3).to_string(index=False))

                Role               Model  Accuracy  Precision  Recall    F1
Published in article       Decision Tree     0.740      0.420   0.410 0.410
Published in article                 KNN     0.730      0.400   0.380 0.390
Published in article                 SVM     0.760      0.480   0.580 0.520
    Our reproduction       Decision Tree     0.735      0.404   0.399 0.401
    Our reproduction                 KNN     0.723      0.379   0.377 0.378
    Our reproduction                 SVM     0.757      0.464   0.566 0.510
  GROUP CONTRIBUTION Logistic Regression     0.696      0.390   0.645 0.486


### Final results at a glance

| Role | Model | Accuracy | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| Published in article | Decision Tree | 0.740 | 0.420 | 0.410 | 0.410 |
| Published in article | KNN | 0.730 | 0.400 | 0.380 | 0.390 |
| Published in article | **SVM** | **0.760** | **0.480** | 0.580 | **0.520** |
| Our reproduction | Decision Tree | 0.735 | 0.404 | 0.399 | 0.401 |
| Our reproduction | KNN | 0.723 | 0.379 | 0.377 | 0.378 |
| Our reproduction | **SVM** | **0.757** | **0.464** | 0.566 | **0.510** |
| **GROUP CONTRIBUTION** | **Logistic Regression** | 0.696 | 0.390 | **0.645** | 0.486 |

The bold values make the main result clear: **SVM has the best overall F1-score, while Logistic Regression has the best recall in our experiment.**

In [14]:
# Compare Logistic Regression directly with the reproduced SVM
reproduced_svm = reproduction_results.loc[
    reproduction_results["Model"] == "SVM"
].iloc[0]
logistic_values = logistic_result.iloc[0]

contribution_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Reproduced SVM": [
        reproduced_svm["Accuracy"],
        reproduced_svm["Precision"],
        reproduced_svm["Recall"],
        reproduced_svm["F1"]
    ],
    "Logistic Regression": [
        logistic_values["Accuracy"],
        logistic_values["Precision"],
        logistic_values["Recall"],
        logistic_values["F1"]
    ]
})
contribution_comparison["Change with Logistic Regression"] = (
    contribution_comparison["Logistic Regression"]
    - contribution_comparison["Reproduced SVM"]
)

print(contribution_comparison.round(3).to_string(index=False))

   Metric  Reproduced SVM  Logistic Regression  Change with Logistic Regression
 Accuracy           0.757                0.696                           -0.062
Precision           0.464                0.390                           -0.074
   Recall           0.566                0.645                            0.079
       F1           0.510                0.486                           -0.024


## 8. Conclusion

The reproduction supports the main finding of the selected article: SVM is the strongest of the three published models for this experiment. Its reproduced F1-score is 0.510, close to the article's reported value of 0.520.

The group's Logistic Regression contribution produced:

- Accuracy: **0.696**
- Precision: **0.390**
- Recall: **0.645**
- F1-score: **0.486**

Logistic Regression did not improve the overall F1-score compared with SVM. However, it improved recall by approximately **0.079** compared with the reproduced SVM. This means it identified more real default cases, but it also generated more false positives.

Therefore, Logistic Regression is a useful contribution when the business priority is to miss fewer high-risk clients. SVM remains the better balanced model according to F1-score.

## Challenges and Learning

- The dataset is imbalanced, so accuracy alone can be misleading.
- The article does not report its random split seed, so exact reproduction is not possible.
- SVM requires much more training time than the other models.
- Oversampling improves minority-class representation but can increase false positives.
- We learned that a new model can be valuable even when it does not produce the highest F1-score, because different business problems may prioritize recall or precision.

## Reference

Ly, T., Schuller, R., Simanic, K., & Singh, S. *Predicting Default of Credit Card Clients Using Three Supervised Machine Learning Algorithms*. MLArticle(1).pdf.